# 02 — Your first C-1N policy

We use the existing C-1N model. The robot's physical geometry, masses, contacts, actuators, and controller implementation stay unchanged. The learning code chooses target commands through `LearningSimulation`.

You described a policy as a parameterized treatment. The next step is to make the treatment depend on what the robot observes. Then we will use experience to change the policy parameters.

The last notebook applied one fixed hip target. A feedback policy makes a new choice from an observation. A stochastic policy supplies probabilities for those choices. REINFORCE is the learning rule we will build to update those probabilities from recorded experience.

We pair in the conversation. Your first attempt can be words or code. This notebook has no prediction form or participation switch. Its supplied cells only prepare and inspect the reset state.

In [ ]:
from pathlib import Path
from dataclasses import fields
import sys

REPO_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent)
     if (path / "learning_env.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start the kernel in spider/ or spider/notebooks/.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from learning_env import LearningSimulation
from simulation import MeasuredState
from notebook_viewer import TreatmentReplay

sim = LearningSimulation()
measured = sim.reset()
print("Physics timestep (s):", sim.model.opt.timestep)
print("Actuators:", [sim.model.actuator(i).name for i in range(sim.model.nu)])
print("Available measurements:", [field.name for field in fields(MeasuredState)])

## First: make one decision

The adapter accepts **18 joint-target offsets**, in radians, relative to the neutral stance. An offset is a command; it is not the resulting joint angle. The number of physics steps for which we hold each choice is also ours to decide.

For a small first exercise, we can let the policy choose between two treatments you have already inspected: neutral targets, or the front-left hip offset of +0.4 radians. This is a proposed teaching simplification. It is not a walking solution or a fixed choice for the later policy.

Propose one **if/then rule** that uses a measurement to choose between those targets. Name the measurement and explain what the rule should make the leg do. An imperfect rule is useful: we will inspect it before adding learning.

Some available measurements:

| Field | Meaning |
| --- | --- |
| `joint_positions` | Current joint angles, in radians |
| `joint_velocities` | Current joint angular velocities, in radians per second |
| `torso_position` | Torso position in world coordinates, in metres |
| `torso_orientation` | Torso orientation as a quaternion |
| `foot_contacts` | Names of feet with declared ground contact |

`MeasuredState` is the available sensor data. The **observation** is the subset or transformation you choose to give the policy. We do not need to feed every measurement into it.

In [ ]:
# Write your first observation-dependent choice after we discuss your rule.
# Keep the policy implementation yours; setup above is ready.

## Then: turn the decision into learning

After your first rule and visible rollout, we will work through one piece at a time:

1. Replace the fixed choice rule with a small stochastic policy.
2. Record the choices, their probabilities, and the resulting rewards.
3. Compute returns and write one REINFORCE update in PyTorch.
4. Compare recorded behavior before and after learning. Keep the physical model fixed.

You already identified that rewarding absolute velocity could favor shuffling without travel. Preserve that distinction when we design the forward-progress objective. We will choose the actual reward and episode boundaries together before training.

The viewer can replay captured states at interpretation time. A starting-pose ghost is only a pose reference. Comparisons of policies need their own recorded rollouts under matched conditions.

## References to use beside our code

- [Spinning Up: implementing the simplest policy gradient](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html#implementing-the-simplest-policy-gradient): connect the learning rule to a small implementation.
- [PyTorch REINFORCE example](https://github.com/pytorch/examples/blob/main/reinforcement_learning/reinforce.py): a compact reference for discrete choices.
- [Gymnasium: REINFORCE in MuJoCo](https://gymnasium.farama.org/tutorials/training_agents/mujoco_reinforce/): the later transition to continuous actions.

Use these as references for each part we reach. The policy, rollout logic, returns, and update in this notebook remain your implementation.